# 🎭 ChuckleNet: Streaming F0 Extraction (No per-utterance I/O)

**Problem**: Per-utterance ffmpeg calls slow in Colab due to Drive I/O.
**Solution**: Load audio ONCE per video, extract all segments in memory.

**Speed**: ~30s per video (vs 9.94s on local SSD)
**Total**: ~5 hours for 620 videos (acceptable for overnight run)

In [ ]:
# 1. Setup
!apt-get install -y ffmpeg 2>&1 | tail -1
!pip install librosa numpy pandas scikit-learn tqdm 2>&1 | tail -3

from google.colab import drive
drive.mount('/content/drive')

import os, glob, time
import numpy as np
import librosa
from tqdm import tqdm

for BASE in ['/content/drive/My Drive/chuckle_net', '/content/drive/Shareddrives/chuckle_net']:
    if os.path.exists(BASE): break

AUDIO_DIR = f'{BASE}/audio'
VTT_DIR = f'{BASE}/vtt'

audio_files = glob.glob(f'{AUDIO_DIR}/*.m4a') + glob.glob(f'{AUDIO_DIR}/*.wav')
vtt_files = glob.glob(f'{VTT_DIR}/*.vtt')
print(f'Audio: {len(audio_files)}, VTT: {len(vtt_files)}')

In [ ]:
# 2. Pre-compute timing for ALL videos
def get_vid(name):
    return name.replace('.en.vtt','').replace('.vtt','').replace('.m4a','').replace('.wav','')

def parse_vtt_cues(vtt_path):
    with open(vtt_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    def to_sec(ts):
        p = ts.replace('.',':').split(':')
        return int(p[0])*3600 + int(p[1])*60 + float(p[2])
    cues, lines = [], content.split('\n')
    i = 0
    while i < len(lines):
        if '-->' in lines[i]:
            s, e = lines[i].split('-->')
            s, e = to_sec(s.strip()), to_sec(e.strip())
            txt, i = [], i+1
            while i < len(lines) and lines[i].strip() and '-->' not in lines[i]:
                txt.append(lines[i].strip()); i += 1
            cues.append((s, e, '[laughter]' in ' '.join(txt).lower()))
        else: i += 1
    return cues

vtt_lookup = {get_vid(os.path.basename(v)): v for v in vtt_files}
audio_lookup = {get_vid(os.path.basename(a)): a for a in audio_files}
matching = list(set(audio_lookup.keys()) & set(vtt_lookup.keys()))

# Pre-count total utterances
total_utts = 0
for vid in tqdm(matching, desc='Counting utterances'):
    cues = parse_vtt_cues(vtt_lookup[vid])
    total_utts += len(cues)

print(f'Total videos: {len(matching)}')
print(f'Total utterances: {total_utts}')

In [ ]:
# 3. STREAMING approach: Load audio ONCE, extract all segments in memory
sr = 22050
hop_length = 512

def extract_features_from_y(y, start_sec, end_sec):
    """Extract piptrack features from pre-loaded audio."""
    start_sample = int(start_sec * sr)
    end_sample = int(end_sec * sr)
    if end_sample > len(y): end_sample = len(y)
    if start_sample >= end_sample: return None
    
    y_seg = y[start_sample:end_sample]
    if len(y_seg) < sr * 0.05: return None
    
    try:
        pitches, magnitudes = librosa.piptrack(y=y_seg, sr=sr, hop_length=hop_length)
        
        # Extract pitch values
        n_frames = min(pitches.shape[1], len(y_seg) // hop_length)
        f0_values = []
        for i in range(n_frames):
            idx = magnitudes[:, i].argmax()
            freq = pitches[idx, i]
            if freq > 0:
                f0_values.append(freq * sr / 2048)
            else:
                f0_values.append(0)
        f0_values = np.array(f0_values)
        voiced = f0_values > 0
        
        return [
            float(np.mean(f0_values)),
            float(np.std(f0_values[voiced])) if voiced.any() else 0,
            float(np.max(f0_values)),
            float(np.min(f0_values[voiced])) if voiced.any() else 0,
            float(np.mean(voiced))
        ]
    except:
        return None

# Test on one video
vid = matching[0]
print(f'Testing on {vid}...')

t0 = time.time()

# Load audio ONCE
audio_path = audio_lookup[vid]
y, _ = librosa.load(audio_path, sr=sr)
t_load = time.time() - t0
print(f'Load: {t_load:.1f}s for {len(y)/sr:.0f}s audio')

# Extract all segments
cues = parse_vtt_cues(vtt_lookup[vid])
t1 = time.time()
for s, e, _ in cues:
    extract_features_from_y(y, s, e)
t_extract = time.time() - t1
print(f'Extract {len(cues)} utterances: {t_extract:.2f}s')
print(f'Per utterance: {1000*t_extract/len(cues):.1f}ms')
print(f'Estimated for {len(matching)} videos: {t_load + t_extract} x {len(matching)} / 60:.0f min')

In [ ]:
# 4. Process ALL 620 videos (streaming)
all_feat, all_label, all_vid, all_lang = [], [], [], []

for vid in tqdm(matching, desc='Videos'):
    audio_path = audio_lookup[vid]
    vtt_path = vtt_lookup[vid]
    
    lang = 'en'
    if '.hi.' in vtt_path: lang = 'hi'
    elif '.zh.' in vtt_path: lang = 'zh'
    elif '.es.' in vtt_path: lang = 'es'
    
    cues = parse_vtt_cues(vtt_path)
    
    # Load audio ONCE
    try:
        y, _ = librosa.load(audio_path, sr=sr)
    except:
        continue
    
    for i, (s, e, has_laugh) in enumerate(cues):
        feat = extract_features_from_y(y, s, e)
        if feat:
            all_feat.append(feat)
            all_label.append(1 if has_laugh else 0)
            all_vid.append(vid)
            all_lang.append(lang)

X = np.array(all_feat)
y = np.array(all_label)
vids = np.array(all_vid)
langs = np.array(all_lang)

print(f'\n✅ Total: {len(X)} segments')
print(f'Positive: {y.sum()} ({100*y.mean():.1f}%)')

In [ ]:
# 5. Train + Evaluate
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

unique_vids = list(set(vids))
np.random.seed(42); np.random.shuffle(unique_vids)
n_test = max(1, int(len(unique_vids)*0.2))
test_vids = set(unique_vids[:n_test])

train_m = ~np.isin(vids, list(test_vids))
X_train, X_test = X[train_m], X[~train_m]
y_train, y_test = y[train_m], y[~train_m]

print(f'Train: {len(X_train)} ({100*y_train.mean():.1f}% pos)\n')

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print('=== Streaming piptrack F0 ===')
print(f'F1: {f1_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall: {recall_score(y_test, y_pred):.4f}')

In [ ]:
# 6. Save
import pickle
out = {'features': X, 'labels': y, 'vids': vids, 'langs': langs}
np.savez_compressed(f'{BASE}/streaming_620.npz', **out)
with open(f'{BASE}/streaming_model.pkl', 'wb') as f:
    pickle.dump(lr, f)
print(f'Saved: {BASE}/streaming_620.npz')
print('\n🎉 DONE!')